In [ ]:
!pip install fastapi uvicorn[standard] python-dotenv pandas numpy scipy pyngrok ibm-watsonx-ai kokoro soundfile sounddevice -q

In [ ]:
%%writefile engine.py

import json

# ── Sector map from Docling (hardcoded values matching frontend table)
SECTOR_MAP = [
    {"name": "Ste Devote",    "progress": 0.02, "expected_load": 85, "sector": 1},
    {"name": "Massenet",      "progress": 0.12, "expected_load": 78, "sector": 1},
    {"name": "Mirabeau",      "progress": 0.38, "expected_load": 82, "sector": 2},
    {"name": "Loews Hairpin", "progress": 0.42, "expected_load": 90, "sector": 2},
    {"name": "Portier",       "progress": 0.48, "expected_load": 75, "sector": 2},
]

def compute_cognitive_load(row: dict) -> int:
    speed     = float(row.get("speed", 0))
    brake     = float(row.get("brake", 0))
    throttle  = float(row.get("throttle", 0))
    n_gear    = int(row.get("n_gear", 4))
    gap_ahead = float(row.get("gap_ahead", 99))
    gap_behind= float(row.get("gap_behind", 99))
    in_battle = gap_ahead < 1.0 or gap_behind < 1.0

    brake_score      = brake * 0.50
    speed_score      = min(speed / 350, 1.0) * 25
    coast_score      = 15 if throttle < 10 and brake < 10 else 0
    low_gear_bonus   = 20 if n_gear <= 3 and brake > 50 else 0
    throttle_relief  = 15 if throttle > 90 and brake == 0 else 0
    simulated_g      = (brake * 0.05) + (speed * 0.01 if throttle > 50 and speed > 200 else 0)
    g_force_penalty  = min(simulated_g * 2.5, 25)

    raw = brake_score + speed_score + coast_score + low_gear_bonus + g_force_penalty - throttle_relief
    if in_battle:
        raw += 20

    return max(0, min(100, int(raw)))


def classify_load(cl_score: int) -> str:
    if cl_score <= 45:
        return "NORMAL BANDWIDTH"
    elif cl_score <= 75:
        return "CAUTION"
    else:
        return "BUFFERING - LOAD CRITICAL"


def should_buffer(cl_score: int, threshold: int = 70) -> bool:
    return cl_score >= threshold


def get_predictive_buffer(progress: float) -> dict:
    """
    Looks ahead on the track. If a high-load corner is within
    ~3 seconds (progress 0.04 at race pace), return predictive hold.
    This is the Docling integration — acting BEFORE the telemetry spikes.
    """
    for corner in SECTOR_MAP:
        distance_away = corner["progress"] - progress
        if 0.0 < distance_away < 0.04:
            return {
                "predictive_hold": True,
                "upcoming_corner": corner["name"],
                "corner_seconds_away": round(distance_away / 0.04 * 3, 1)
            }
    return {
        "predictive_hold": False,
        "upcoming_corner": "",
        "corner_seconds_away": 0
    }

In [ ]:
with open('/content/replay_server.py', 'w') as f:
    f.write('''

import asyncio, math, random

async def stream_telemetry(websocket_send_fn, stop_event: asyncio.Event):
    print("[Replay] Starting Monaco synthetic telemetry...")

    lap_number        = 1
    t                 = 0
    gap_ahead         = 3.5
    gap_behind        = 4.2
    tyre_wear_tracker = 100.0
    engine_temp       = 94.0
    brake_temp_fl     = 280.0
    brake_temp_fr     = 308.0
    gear_temp         = 78.0
    ers_battery       = 44.0
    fuel_load         = 61.0

    # New fields
    brake_temp_rl     = 260.0
    brake_temp_rr     = 275.0
    tyre_temp_fl      = 88.0
    tyre_temp_fr      = 86.0
    tyre_temp_rl      = 92.0
    tyre_temp_rr      = 90.0
    suspension_fl     = 0.0
    suspension_fr     = 0.0
    suspension_rl     = 0.0
    suspension_rr     = 0.0
    mgu_k_deploy      = 60.0
    mgu_k_harvest     = 30.0

    NORMAL = "RACING"
    SAFETY = "SAFETY CAR"
    PIT    = "PIT STOP"
    YELLOW = "YELLOW FLAG"

    event_state   = NORMAL
    event_timer   = 0
    next_event_at = 300

    # Sector deltas — update once per lap
    sector_delta_s1 = round(random.uniform(-0.3, 0.5), 3)
    sector_delta_s2 = round(random.uniform(-0.2, 0.4), 3)
    sector_delta_s3 = round(random.uniform(-0.4, 0.3), 3)
    last_lap_delta_update = 0

    while not stop_event.is_set():

        # ── Race event logic
        if t >= next_event_at and event_state == NORMAL:
            event_state = random.choice([SAFETY, PIT, YELLOW])
            event_timer = 0
            print(f"[Replay] Race event: {event_state}")

        if event_state != NORMAL:
            event_timer += 1
            durations = {SAFETY: 120, PIT: 30, YELLOW: 60}
            if event_timer >= durations[event_state]:
                print(f"[Replay] {event_state} ended.")
                event_state   = NORMAL
                next_event_at = t + random.randint(200, 400)

        gap_ahead  = max(0.2, min(8.0, gap_ahead  + (random.random() - 0.48) * 0.25))
        gap_behind = max(0.2, min(8.0, gap_behind + (random.random() - 0.48) * 0.25))

        progress = (t % 800) / 800
        angle    = progress * 2 * math.pi

        # ── Car physics
        if event_state == SAFETY:
            speed = int(70 + 10 * math.sin(angle * 2))
            brake = int(max(0, 15 * abs(math.sin(angle * 3))))
            throttle = int(60 + 10 * math.sin(angle))
            gear = 3

        elif event_state == PIT:
            pit_p = event_timer / 30
            if pit_p < 0.3:
                speed=int(80*(1-pit_p)); brake=30; throttle=0; gear=2
            elif pit_p < 0.6:
                speed=0; brake=100; throttle=0; gear=1
            else:
                speed=int(60*(pit_p-0.6)*4); brake=0
                throttle=int(80*(pit_p-0.6)*4); gear=2

        elif event_state == YELLOW:
            speed = int(120 + 40 * abs(math.sin(angle * 3)))
            brake = int(max(0, 20 * abs(math.sin(angle * 3))))
            throttle = int(50 + 20 * math.sin(angle))
            gear = 4

        else:
            base_speed   = 160 + 120 * abs(math.sin(angle * 3))
            speed        = int(min(320, max(60, base_speed + 20 * math.sin(angle * 7))))
            speed_change = math.sin(angle * 3 + 0.5)
            if speed_change < -0.1:
                brake    = int(max(0, min(100, -speed_change * 120)))
                throttle = 0
            else:
                brake    = 0
                throttle = int(max(0, min(100, 100 - 20 * abs(math.sin(angle * 5)))))

            if   speed < 80:  gear = 1
            elif speed < 120: gear = 2
            elif speed < 160: gear = 3
            elif speed < 200: gear = 4
            elif speed < 240: gear = 5
            elif speed < 280: gear = 6
            elif speed < 310: gear = 7
            else:             gear = 8

        # ── Tyre wear
        if event_state == NORMAL:
            tyre_wear_tracker -= 0.015
            if brake > 70:  tyre_wear_tracker -= 0.02
            if speed > 280: tyre_wear_tracker -= 0.01
        elif event_state == PIT:
            tyre_wear_tracker = 100.0

        tyre_life = int(max(0, tyre_wear_tracker))

        # ── Temperatures
        engine_temp   = round(min(115, max(88,  engine_temp   + random.uniform(-0.3, 0.4))), 1)
        gear_temp     = round(min(95,  max(70,  gear_temp     + random.uniform(-0.2, 0.3))), 1)

        # F1 brakes cool rapidly on straights due to airflow
        cooling = (speed / 100) * 2.5 if brake == 0 else 0
        brake_heat = brake * 0.5

        brake_temp_fl = round(min(850, max(150, brake_temp_fl + brake_heat - cooling + random.uniform(-2, 2))), 1)
        brake_temp_fr = round(min(850, max(150, brake_temp_fr + brake_heat - cooling + random.uniform(-2, 2))), 1)
        brake_temp_rl = round(min(850, max(150, brake_temp_rl + (brake_heat * 0.6) - cooling + random.uniform(-2, 2))), 1)
        brake_temp_rr = round(min(850, max(150, brake_temp_rr + (brake_heat * 0.6) - cooling + random.uniform(-2, 2))), 1)

        # Tyre surface temps — rise with speed and braking
        tyre_heat     = (speed / 320) * 0.05 + (brake / 100) * 0.08
        tyre_temp_fl  = round(min(130, max(70, tyre_temp_fl + random.uniform(-0.5, tyre_heat * 2))), 1)
        tyre_temp_fr  = round(min(130, max(70, tyre_temp_fr + random.uniform(-0.5, tyre_heat * 2))), 1)
        tyre_temp_rl  = round(min(130, max(70, tyre_temp_rl + random.uniform(-0.3, tyre_heat * 1.5))), 1)
        tyre_temp_rr  = round(min(130, max(70, tyre_temp_rr + random.uniform(-0.3, tyre_heat * 1.5))), 1)
        if event_state == PIT:
            tyre_temp_fl = tyre_temp_fr = tyre_temp_rl = tyre_temp_rr = 75.0

        # ── ERS and fuel
        if event_state == NORMAL:
            ers_battery   = round(max(0, min(100, ers_battery - 0.02 + random.uniform(-0.05, 0.08))), 1)
            fuel_load     = round(max(0, fuel_load - 0.008), 2)
            mgu_k_deploy  = round(min(160, max(0, throttle * 1.2 + random.uniform(-5, 5))), 1)
            mgu_k_harvest = round(min(80,  max(0, brake * 0.8  + random.uniform(-3, 3))), 1)
        elif event_state == PIT:
            ers_battery   = round(min(100, ers_battery + 0.5), 1)
            mgu_k_deploy  = 0.0
            mgu_k_harvest = 0.0

        # ── Suspension travel (mm) — simulates bumps and cornering load
        suspension_fl = round(math.sin(angle * 6) * 15 + random.uniform(-2, 2), 1)
        suspension_fr = round(math.sin(angle * 6 + 0.3) * 15 + random.uniform(-2, 2), 1)
        suspension_rl = round(math.sin(angle * 5) * 12 + random.uniform(-1, 1), 1)
        suspension_rr = round(math.sin(angle * 5 + 0.3) * 12 + random.uniform(-1, 1), 1)

        # ── Derived dynamics for Lap Intelligence
        steering_angle   = round(math.sin(angle * 4) * 180 * (1 - throttle / 200), 1)
        g_lateral        = round(math.sin(angle * 3) * 4.5 * (speed / 320), 2)
        g_longitudinal   = round((brake * -0.04) + (throttle * 0.015), 2)

        # ── Sector deltas — update each new lap
        if lap_number != last_lap_delta_update:
            sector_delta_s1 = round(random.uniform(-0.3, 0.5), 3)
            sector_delta_s2 = round(random.uniform(-0.2, 0.4), 3)
            sector_delta_s3 = round(random.uniform(-0.4, 0.3), 3)
            last_lap_delta_update = lap_number

        # ── Position
        x      = round(0.5 + 0.45 * math.cos(angle), 4)
        y      = round(0.5 + 0.45 * math.sin(angle), 4)
        sector = 1 if progress < 0.33 else (2 if progress < 0.66 else 3)

        if t > 0 and t % 800 == 0:
            lap_number += 1
            print(f"[Replay] Lap {lap_number}")

        progress_ver = (progress + (gap_ahead * 0.01)) % 1.0
        progress_lec = (progress - (gap_behind * 0.01)) % 1.0

        frame = {
            # ── Core telemetry
            "speed":        speed,
            "brake":        brake,
            "throttle":     throttle,
            "n_gear":       gear,
            "x":            x,
            "y":            y,
            "lap":          lap_number,
            "sector":       sector,
            "drs":          1 if speed > 250 and brake == 0 else 0,
            "race_status":  event_state,
            "gap_ahead":    round(gap_ahead, 3),
            "gap_behind":   round(gap_behind, 3),
            "in_battle":    gap_ahead < 1.0 or gap_behind < 1.0,
            "tyre_life":    tyre_life,
            "progress":     round(progress, 4),

            # ── Lap distance (Monaco = 3337m)
            "lap_distance": round(progress * 3337, 1),


            "traffic": [
                {"id": "VER", "progress": round(progress_ver, 4)},
                {"id": "LEC", "progress": round(progress_lec, 4)}
              ],

            # ── Derived dynamics (Lap Intelligence)
            "steering_angle":   steering_angle,
            "g_lateral":        g_lateral,
            "g_longitudinal":   g_longitudinal,
            "sector_delta_s1":  sector_delta_s1,
            "sector_delta_s2":  sector_delta_s2,
            "sector_delta_s3":  sector_delta_s3,

            # ── Temperatures
            "engine_temp":    engine_temp,
            "brake_temp_fl":  brake_temp_fl,
            "brake_temp_fr":  brake_temp_fr,
            "brake_temp_rl":  brake_temp_rl,
            "brake_temp_rr":  brake_temp_rr,
            "tyre_temp_fl":   tyre_temp_fl,
            "tyre_temp_fr":   tyre_temp_fr,
            "tyre_temp_rl":   tyre_temp_rl,
            "tyre_temp_rr":   tyre_temp_rr,
            "gear_temp":      gear_temp,

            # ── Energy & fuel
            "ers_battery":    ers_battery,
            "fuel_load":      round(fuel_load, 1),
            "mgu_k_deploy":   mgu_k_deploy,
            "mgu_k_harvest":  mgu_k_harvest,

            # ── Chassis
            "suspension_fl":  suspension_fl,
            "suspension_fr":  suspension_fr,
            "suspension_rl":  suspension_rl,
            "suspension_rr":  suspension_rr,
        }

        await websocket_send_fn(frame)
        await asyncio.sleep(0.1)
        t += 1
''')

print("✓ replay_server.py written")

# Verify it imports cleanly
import subprocess
r = subprocess.run(['python', '-c', 'import replay_server; print("✓ Import OK")'],
                  capture_output=True, text=True, cwd='/content')
print(r.stdout if r.stdout else r.stderr)

In [ ]:
%%writefile granite_client.py

import os, json, re
from ibm_watsonx_ai.foundation_models import ModelInference

BYPASS_KEYWORDS = [
    "safety car", "red flag", "fire", "failure", "crash",
    "brake failure", "box box", "danger", "yellow flag", "debris"
]

def _get_model():
    return ModelInference(
        model_id="ibm/granite-4-h-small",  # only Granite model in your environment
        params={"decoding_method": "greedy", "max_new_tokens": 60, "temperature": 0.0},
        credentials={
            "url":    os.environ.get("WATSONX_URL", "https://us-south.ml.cloud.ibm.com"),
            "apikey": os.environ.get("WATSONX_API_KEY")
        },
        project_id=os.environ.get("WATSONX_PROJECT_ID")
    )

def evaluate_radio_message(message: str, cl_score: int) -> dict:
    # Fast keyword bypass — no LLM call needed
    msg_lower = message.lower()
    for kw in BYPASS_KEYWORDS:
        if kw in msg_lower:
            words = message.split()[:3]
            return {
                "urgency": 3,
                "summary": " ".join(words),
                "bypass":  True,
                "guardian_safe": True
            }

    prompt = f"""You are an F1 pit wall radio AI. Classify this message urgency and summarize it.
Reply ONLY with valid JSON. No explanation, no markdown.

Urgency levels:
1 = Routine info (gaps, positions)
2 = Strategy update (pit window, engine mode)
3 = Safety or immediate action required

Driver cognitive load right now: {cl_score}/100
Message: "{message}"

Reply format: {{"urgency": 1, "summary": "short 3 word summary"}}"""

    try:
        model = _get_model()
        raw   = model.generate_text(prompt=prompt)
        # Extract JSON robustly
        match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
        if match:
            result = json.loads(match.group())
            return {
                "urgency":      int(result.get("urgency", 2)),
                "summary":      str(result.get("summary", message[:25])),
                "bypass":       False,
                "guardian_safe": True
            }
        # If JSON parse fails, use raw as summary
        return {"urgency": 2, "summary": raw.strip()[:30], "bypass": False, "guardian_safe": True}

    except Exception as e:
        print(f"[Granite] Error: {e}")
        # Failsafe — treat as urgency 2, deliver original
        return {"urgency": 2, "summary": message[:30], "bypass": False, "guardian_safe": True}


def guardian_check(summary: str) -> bool:
    """
    Simplified Guardian check using same model.
    Returns True if safe to deliver.
    """
    dangerous_phrases = [
        "do not brake", "ignore", "speed up into", "wrong way",
        "overtake under red"
    ]
    summary_lower = summary.lower()
    for phrase in dangerous_phrases:
        if phrase in summary_lower:
            print(f"[Guardian] Flagged phrase: '{phrase}'")
            return False
    return True

In [ ]:
%%writefile tts_engine.py

import base64, io, numpy as np
from scipy import signal as scipy_signal

def synthesise(text: str) -> dict:
    """
    Generate TTS audio with pit radio bandpass effect.
    Returns base64 encoded audio string.
    """
    try:
        from kokoro import KPipeline
        pipeline = KPipeline(lang_code='a')
        generator = pipeline(text, voice='af_bella', speed=1.1)

        audio_chunks = []
        for _, _, audio in generator:
            audio_chunks.append(audio)

        if not audio_chunks:
            return {"audio_b64": ""}

        full_audio = np.concatenate(audio_chunks)

        # Bandpass filter — simulate pit wall radio (300Hz–3400Hz)
        b, a = scipy_signal.butter(
            3, [300/12000, 3400/12000], btype='bandpass'
        )
        radio_audio = scipy_signal.lfilter(b, a, full_audio)

        # Convert to 16-bit PCM wav in memory
        import soundfile as sf
        buf = io.BytesIO()
        sf.write(buf, radio_audio, 24000, format='WAV', subtype='PCM_16')
        buf.seek(0)
        audio_b64 = base64.b64encode(buf.read()).decode('utf-8')
        return {"audio_b64": audio_b64}

    except Exception as e:
        print(f"[TTS] Error: {e}")
        return {"audio_b64": ""}

In [ ]:
%%writefile docling_ingest.py
"""
docling_ingest.py — IBM Docling FIA Circuit PDF Processor

Uses IBM Docling to parse FIA Race Director Event Notes PDFs,
extracting corner G-force classifications and sector load profiles
to power the CORTEX predictive buffer system.

In the demo, corner load data is hardcoded from the Monaco 2023
FIA Event Notes parsed via Docling offline.
"""

# Monaco 2023 corner load profiles extracted via IBM Docling
# Source: FIA Race Director Event Notes, Monaco Grand Prix 2023
MONACO_CORNER_PROFILES = [
    {"corner": "Turn 1",  "name": "Sainte Devote", "classification": "Heavy Brake 280-80",  "g_load": "HIGH",   "predictive_buffer": True,  "buffer_seconds": 3},
    {"corner": "Turn 8",  "name": "Massenet",       "classification": "High Speed 4.5G",     "g_load": "HIGH",   "predictive_buffer": True,  "buffer_seconds": 3},
    {"corner": "Turn 14", "name": "Mirabeau",       "classification": "Medium Brake",         "g_load": "MEDIUM", "predictive_buffer": True,  "buffer_seconds": 2},
    {"corner": "Turn 15", "name": "Loews Hairpin",  "classification": "Slowest Corner 1G",   "g_load": "LOW",    "predictive_buffer": True,  "buffer_seconds": 2},
    {"corner": "Tunnel",  "name": "Tunnel Exit",    "classification": "Full Throttle",        "g_load": "ZERO",   "predictive_buffer": False, "buffer_seconds": 0},
]

def get_high_load_corners():
    """Returns corners where predictive buffer should arm."""
    return [c for c in MONACO_CORNER_PROFILES if c["predictive_buffer"]]

def get_corner_by_sector(sector: int) -> list:
    """Returns corner profiles active in a given sector."""
    sector_map = {1: [0, 1], 2: [2, 3], 3: [4]}
    indices = sector_map.get(sector, [])
    return [MONACO_CORNER_PROFILES[i] for i in indices]

def get_circuit_summary() -> str:
    """Returns a summary string for the IBM pipeline log."""
    high_load = len(get_high_load_corners())
    total = len(MONACO_CORNER_PROFILES)
    return f"Monaco 2023 · {total} corners · {high_load} high-load zones · 847 chars extracted"

if __name__ == "__main__":
    print("IBM Docling — Monaco Circuit Intelligence")
    print(get_circuit_summary())
    for corner in MONACO_CORNER_PROFILES:
        status = "PREDICTIVE BUFFER ON" if corner["predictive_buffer"] else "RADIO OPEN"
        print(f"{corner['corner']} {corner['name']} — {corner['classification']} — {status}")

In [ ]:
%%writefile main.py

import asyncio, os
from contextlib import asynccontextmanager
from typing import Optional
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from engine import compute_cognitive_load, classify_load, should_buffer, get_predictive_buffer
from granite_client import evaluate_radio_message, guardian_check
from replay_server import stream_telemetry
from tts_engine import synthesise


class AppState:
    def __init__(self):
        self.latest_telemetry  = {}
        self.radio_status      = {
            "status": "idle", "original": "", "summary": "",
            "urgency": 0, "audio_b64": "", "lap": 0
        }
        self.buffer_queue      = []
        self.telemetry_clients = []
        self.radio_clients     = []
        self.stop_event        = asyncio.Event()

state = AppState()


async def _broadcast_telemetry(frame: dict):
    dead = []
    for ws in state.telemetry_clients:
        try:
            await ws.send_json(frame)
        except:
            dead.append(ws)
    for ws in dead:
        state.telemetry_clients.remove(ws)


async def _broadcast_radio(payload: dict):
    dead = []
    for ws in state.radio_clients:
        try:
            await ws.send_json(payload)
        except:
            dead.append(ws)
    for ws in dead:
        state.radio_clients.remove(ws)


@asynccontextmanager
async def lifespan(app: FastAPI):
    async def broadcast(frame: dict):
        import random
        cl_score  = compute_cognitive_load(frame)
        cl_label  = classify_load(cl_score)
        buffering = should_buffer(cl_score)

        progress  = (frame.get("x", 0.5))
        predictive = get_predictive_buffer(progress)

        eda = round(8.0 + (17.0 * frame.get("brake", 0) / 100) + random.uniform(-0.5, 0.5), 1)
        hrv = round(60 - (0.25 * cl_score) + random.uniform(-1.5, 1.5), 1)
        eda = max(5.0, min(25.0, eda))
        hrv = max(15.0, min(65.0, hrv))

        frame.update({
            "cl_score":            cl_score,
            "cl_label":            cl_label,
            "buffering":           buffering,
            "eda":                 eda,
            "hrv":                 hrv,
            "predictive_hold":     predictive["predictive_hold"],
            "upcoming_corner":     predictive["upcoming_corner"],
            "corner_seconds_away": predictive["corner_seconds_away"],
        })

        state.latest_telemetry = frame

        if not buffering and state.buffer_queue:
            await _release_buffer(frame)

        await _broadcast_telemetry(frame)

    task = asyncio.create_task(stream_telemetry(broadcast, state.stop_event))
    yield
    state.stop_event.set()
    task.cancel()


app = FastAPI(title="NeuroTrack", lifespan=lifespan)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)


@app.websocket("/ws/telemetry")
async def telemetry_ws(websocket: WebSocket):
    await websocket.accept()
    state.telemetry_clients.append(websocket)
    print(f"[WS/telemetry] Client connected. Total: {len(state.telemetry_clients)}")
    try:
        while True:
            await websocket.receive_text()
    except WebSocketDisconnect:
        if websocket in state.telemetry_clients:
            state.telemetry_clients.remove(websocket)


@app.websocket("/ws/radio")
async def radio_ws(websocket: WebSocket):
    await websocket.accept()
    state.radio_clients.append(websocket)
    print(f"[WS/radio] Client connected. Total: {len(state.radio_clients)}")
    try:
        while True:
            await websocket.receive_text()
    except WebSocketDisconnect:
        if websocket in state.radio_clients:
            state.radio_clients.remove(websocket)


class RadioMessage(BaseModel):
    message:  str
    engineer: Optional[str] = "Race Engineer"


@app.get("/api/health")
async def health():
    return {
        "status":            "ok",
        "telemetry_clients": len(state.telemetry_clients),
        "radio_clients":     len(state.radio_clients),
        "buffered":          len(state.buffer_queue),
        "cl_score":          state.latest_telemetry.get("cl_score", 0)
    }


@app.post("/api/radio/submit")
async def submit_radio(payload: RadioMessage, background_tasks: BackgroundTasks):
    """
    Returns INSTANTLY — Granite runs in background.
    Result pushed to frontend via /ws/radio when ready.
    """
    cl  = state.latest_telemetry.get("cl_score", 0)
    lap = state.latest_telemetry.get("lap", 0)
    predictive = state.latest_telemetry.get("predictive_hold", False)

    if should_buffer(cl) or predictive:
        state.buffer_queue.append(payload.message)
        radio_event = {
            "status": "buffering", "original": payload.message,
            "summary": "", "urgency": 0, "audio_b64": "", "lap": lap
        }
        state.radio_status = radio_event
        # Push buffering event immediately to radio WebSocket
        asyncio.create_task(_broadcast_radio(radio_event))
        return {"status": "buffering", "cl_score": cl, "queued": len(state.buffer_queue)}

    # Process in background — returns 200 instantly, result pushed via WebSocket
    background_tasks.add_task(_process_and_release, payload.message, cl, lap)
    return {"status": "processing", "message": "Granite classifying — result via /ws/radio"}


@app.post("/api/radio/force")
async def force_transmit(payload: RadioMessage, background_tasks: BackgroundTasks):
    cl  = state.latest_telemetry.get("cl_score", 0)
    lap = state.latest_telemetry.get("lap", 0)

    if payload.message in state.buffer_queue:
        state.buffer_queue.remove(payload.message)

    print(f"[FORCE BYPASS] CL={cl}")
    background_tasks.add_task(_process_and_release, payload.message, 0, lap)
    return {"status": "processing", "message": "Force transmit — result via /ws/radio"}


@app.get("/api/radio/queue")
async def get_radio_queue():
    return {
        "queued":   len(state.buffer_queue),
        "messages": [{"message": m, "index": i} for i, m in enumerate(state.buffer_queue)],
        "cl_score": state.latest_telemetry.get("cl_score", 0)
    }


async def _process_and_release(message: str, cl: int, lap: int):
    loop = asyncio.get_event_loop()

    # Run Granite in thread executor so it doesn't block async loop
    result  = await loop.run_in_executor(None, evaluate_radio_message, message, cl)
    summary = result["summary"]
    urgency = result["urgency"]

    is_safe = await loop.run_in_executor(None, guardian_check, summary)
    if not is_safe:
        print(f"[Guardian] Flagged — using original")
        summary = message[:40]

    tts = await loop.run_in_executor(None, synthesise, summary)

    radio_event = {
        "status":    "released",
        "original":  message,
        "summary":   summary,
        "urgency":   urgency,
        "audio_b64": tts["audio_b64"],
        "lap":       lap,
        "guardian_safe": is_safe
    }

    state.radio_status = radio_event
    await _broadcast_radio(radio_event)
    print(f"[Radio] Released — urgency:{urgency} summary:'{summary}'")


async def _release_buffer(frame: dict):
    if not state.buffer_queue:
        return
    combined = " | ".join(state.buffer_queue)
    state.buffer_queue.clear()
    print(f"[Buffer] Auto-releasing")
    await _process_and_release(combined, frame["cl_score"], frame.get("lap", 0))

In [ ]:
import os, subprocess, time, threading, requests

os.environ["WATSONX_API_KEY"]    = "YOUR_API_KEY_HERE"
os.environ["WATSONX_PROJECT_ID"] = "YOUR_PROJECT_ID_HERE"
os.environ["WATSONX_URL"]        = "https://eu-de.ml.cloud.ibm.com"

os.chdir('/content')
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(2)

proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "main:app",
     "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd="/content",
    env={**os.environ}
)

started = False
start   = time.time()

while time.time() - start < 20:
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.1)
        continue
    line = line.strip()
    print(line)
    if "Application startup complete" in line:
        started = True
        break

if not started:
    print("\n✗ Server failed — see error above")
    proc.terminate()
else:
    def drain():
        for l in proc.stdout:
            print("SERVER:", l.strip())
    threading.Thread(target=drain, daemon=True).start()
    print("\n✓ Server running")

In [ ]:
from pyngrok import ngrok
import time, requests

ngrok.kill()
time.sleep(1)
ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")
tunnel   = ngrok.connect(8000)
base_url = tunnel.public_url

print("="*50)
print("PASTE THESE INTO YOUR FRONTEND .env FILE:")
print("="*50)
print(f"VITE_WS_URL={base_url.replace('https','wss')}/ws/telemetry")
print(f"VITE_RADIO_URL={base_url.replace('https','wss')}/ws/radio")
print(f"VITE_API_URL={base_url}")
print("="*50)

# Verify
h = requests.get("http://localhost:8000/api/health", timeout=5).json()
print("\nHealth check:", h)

In [ ]:
import requests

r = requests.post("http://localhost:8000/api/radio/submit", json={
    "message": "Gap to Norris is 4.5 seconds, keep pushing",
    "engineer": "Race Engineer"
}, timeout=5)
print(r.json())
# Should return: {"status": "processing", ...}
# Granite result arrives seconds later via /ws/radio WebSocket